# 네이버 블로그/카페글 URL 원문 보강

`crawl_naver_blog_cafe.ipynb`에서 만든 주제별 CSV(`crawl_data_비염.csv` 등)의 URL에 접속해 가능한 경우 전체 본문과 댓글을 수집합니다.

주의할 점:
- `CORE_KEYWORD`를 바꾸면 해당 주제 CSV만 처리합니다.
- 이 노트북은 한 번에 한 주제 파일만 처리하도록 두었습니다. 비염 데이터에 피부미용 데이터가 섞이지 않게 하기 위한 구조입니다.
- 공개 접근 가능한 글만 수집합니다. 로그인/가입/등급 제한 글은 우회하지 않습니다.
- 블로그는 비교적 수집 성공률이 높을 수 있지만, 카페글은 접근 제한이나 동적 렌더링 때문에 실패할 수 있습니다.
- 먼저 테스트 셀을 실행해 성공률과 본문 품질을 확인한 뒤 전체 수집을 실행하세요.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import time
from datetime import datetime
from html import unescape
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse


import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment

BASE_DIR = Path(os.getenv("DATA_JOUR_BASE_DIR", ".")).expanduser().resolve()

AVAILABLE_CORE_KEYWORDS = ["비염", "탈모", "피부미용", "키성장", "다이어트"]

# 처리할 주제를 하나씩 지정합니다. 예: "비염", "탈모", "피부미용", "키성장", "다이어트"
# 이 값을 바꾸면 아래 입력/백업/댓글 파일 경로가 모두 해당 주제 기준으로 바뀝니다.
CORE_KEYWORD = "피부미용"

if CORE_KEYWORD not in AVAILABLE_CORE_KEYWORDS:
    raise ValueError(f"CORE_KEYWORD는 {AVAILABLE_CORE_KEYWORDS} 중 하나여야 합니다: {CORE_KEYWORD}")


def safe_filename(value):
    value = str(value).strip()
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"\s+", "_", value)
    return value


DATA_FILE_STEM = f"crawl_data_{safe_filename(CORE_KEYWORD)}"
COMMENTS_FILE_STEM = f"crawl_comments_{safe_filename(CORE_KEYWORD)}"

INPUT_CSV = BASE_DIR / f"{DATA_FILE_STEM}.csv"

# 원본 CSV를 직접 갱신하기 전 백업 파일을 만듭니다.
BACKUP_CSV = BASE_DIR / f"{DATA_FILE_STEM}_before_fulltext_backup.csv"
TEMP_OUTPUT_CSV = BASE_DIR / f"{DATA_FILE_STEM}_with_fulltext.tmp.csv"
COMMENTS_CSV = BASE_DIR / f"{COMMENTS_FILE_STEM}.csv"
COMMENTS_TEMP_CSV = BASE_DIR / f"{COMMENTS_FILE_STEM}.tmp.csv"

print(f"처리 주제: {CORE_KEYWORD}")
print(f"입력 CSV: {INPUT_CSV}")
print(f"본문 보강 백업 CSV: {BACKUP_CSV}")
print(f"댓글 CSV: {COMMENTS_CSV}")

# 테스트와 전체 수집 옵션
TEST_ROWS = 5
REQUEST_DELAY_SEC = 0.7
REQUEST_TIMEOUT = 15
SAVE_EVERY = 50
SKIP_EXISTING_FULL_TEXT = True
MIN_TEXT_LENGTH = 80

# 댓글 수집을 중단했다가 다시 실행할 때 이미 처리된 게시글을 건너뜁니다.
SKIP_EXISTING_COMMENTS = True

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

FULLTEXT_COLUMNS = [
    "full_text",
    "full_text_status",
    "full_text_length",
    "full_text_url",
    "full_text_collected_at",
    "full_text_error",
]

COMMENT_SUMMARY_COLUMNS = [
    "comment_count_collected",
    "comment_fetch_status",
    "comments_file_key",
    "comment_collected_at",
]

POST_LINK_COLUMNS = ["link", "게시글URL", "post_link"]
POST_SOURCE_COLUMNS = ["source", "수집출처"]
POST_TITLE_COLUMNS = ["title", "게시글제목", "post_title"]
EXISTING_FULL_TEXT_COLUMNS = ["full_text", "본문전체"]


def first_nonempty(row, *columns):
    for column in columns:
        value = row.get(column, "")
        if value is None:
            continue
        value = str(value).strip()
        if value:
            return value
    return ""


def get_post_link(row):
    return first_nonempty(row, *POST_LINK_COLUMNS)


def get_post_source(row):
    return first_nonempty(row, *POST_SOURCE_COLUMNS)


def get_post_title(row):
    return first_nonempty(row, *POST_TITLE_COLUMNS)


def get_existing_full_text(row):
    return first_nonempty(row, *EXISTING_FULL_TEXT_COLUMNS)


def validate_post_dataframe(df, path):
    if "core_keyword" in df.columns:
        values = sorted(df["core_keyword"].dropna().astype(str).str.strip().unique())
        values = [value for value in values if value]
        if values and values != [CORE_KEYWORD]:
            raise ValueError(
                f"입력 파일에 현재 CORE_KEYWORD와 다른 주제가 섞여 있습니다. "
                f"CORE_KEYWORD={CORE_KEYWORD}, 파일={path}, 파일 내 값={values}"
            )
    if "link" in df.columns:
        duplicate_rows = int(df.duplicated(subset=["link"], keep=False).sum())
        if duplicate_rows:
            print(f"경고: 입력 파일에 URL 중복 행이 {duplicate_rows:,}건 남아 있습니다.")


## 1. CSV 확인

In [ ]:
def read_csv_rows(path):
    if not path.exists():
        raise FileNotFoundError(
            f"입력 파일이 없습니다: {path}. "
            "먼저 test_crawl_teamproject.ipynb에서 해당 주제의 crawl_data_*.csv를 생성하세요."
        )
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")
    validate_post_dataframe(df, path)
    return df.to_dict("records"), list(df.columns)


rows, fieldnames = read_csv_rows(INPUT_CSV)
print(f"입력 파일: {INPUT_CSV}")
print(f"행 수: {len(rows):,}")
print(f"컬럼: {fieldnames}")

for i, row in enumerate(rows[:5], start=1):
    print(f"{i}. [{get_post_source(row)}] {get_post_link(row)}")

## 2. HTML에서 본문 추출하는 함수

`BeautifulSoup`을 사용해 네이버 블로그/카페에서 자주 쓰이는 본문 컨테이너를 먼저 찾고, 실패하면 전체 HTML에서 보이는 텍스트를 추출합니다.

In [ ]:
BLOCK_TAGS = {"p", "div", "br", "li", "section", "article", "h1", "h2", "h3", "h4", "tr"}
SKIP_TAGS = {"script", "style", "noscript", "svg", "canvas"}


def normalize_text(text):
    text = unescape(text or "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return "\n".join(lines).strip()


class IframeSrcParser(HTMLParser):
    def __init__(self, iframe_id_keywords):
        super().__init__()
        self.iframe_id_keywords = iframe_id_keywords
        self.src = None

    def handle_starttag(self, tag, attrs):
        if self.src or tag.lower() != "iframe":
            return
        attrs = dict(attrs)
        iframe_id = attrs.get("id", "")
        iframe_name = attrs.get("name", "")
        haystack = f"{iframe_id} {iframe_name}".lower()
        if any(keyword.lower() in haystack for keyword in self.iframe_id_keywords):
            self.src = attrs.get("src")


class TargetTextParser(HTMLParser):
    def __init__(self, attr_name, keyword):
        super().__init__()
        self.attr_name = attr_name
        self.keyword = keyword.lower()
        self.collecting = False
        self.depth = 0
        self.skip_depth = 0
        self.current = []
        self.segments = []

    def _matches(self, attrs):
        attrs = dict(attrs)
        value = attrs.get(self.attr_name, "")
        return self.keyword in value.lower()

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if self.collecting:
            self.depth += 1
            if tag in BLOCK_TAGS:
                self.current.append("\n")
            if tag in SKIP_TAGS:
                self.skip_depth += 1
            return

        if self._matches(attrs):
            self.collecting = True
            self.depth = 1
            self.skip_depth = 1 if tag in SKIP_TAGS else 0
            self.current = []

    def handle_endtag(self, tag):
        tag = tag.lower()
        if not self.collecting:
            return
        if tag in BLOCK_TAGS:
            self.current.append("\n")
        if tag in SKIP_TAGS and self.skip_depth > 0:
            self.skip_depth -= 1
        self.depth -= 1
        if self.depth <= 0:
            text = normalize_text("".join(self.current))
            if text:
                self.segments.append(text)
            self.collecting = False
            self.current = []

    def handle_data(self, data):
        if self.collecting and self.skip_depth == 0:
            self.current.append(data)


class VisibleTextParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []
        self.skip_depth = 0

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag in BLOCK_TAGS:
            self.parts.append("\n")
        if tag in SKIP_TAGS:
            self.skip_depth += 1

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag in BLOCK_TAGS:
            self.parts.append("\n")
        if tag in SKIP_TAGS and self.skip_depth > 0:
            self.skip_depth -= 1

    def handle_data(self, data):
        if self.skip_depth == 0:
            self.parts.append(data)


def find_iframe_src(html, base_url, iframe_keywords):
    parser = IframeSrcParser(iframe_keywords)
    parser.feed(html)
    if not parser.src:
        return None
    src = parser.src.strip()
    if src.lower().startswith(("javascript:", "about:", "#")):
        return None
    return urljoin(base_url, src)


def extract_target_text(html):
    targets = [
        ("class", "se-main-container"),
        ("id", "postViewArea"),
        ("id", "post-view"),
        ("class", "post_view"),
        ("class", "se_component_wrap"),
        ("class", "ContentRenderer"),
        ("class", "article_viewer"),
        ("class", "ArticleContentBox"),
        ("id", "tbody"),
    ]
    candidates = []
    for attr_name, keyword in targets:
        parser = TargetTextParser(attr_name, keyword)
        try:
            parser.feed(html)
            candidates.extend(parser.segments)
        except Exception:
            continue

    candidates = [text for text in candidates if len(text) >= 20]
    if candidates:
        return max(candidates, key=len)

    parser = VisibleTextParser()
    parser.feed(html)
    return normalize_text("".join(parser.parts))

### BeautifulSoup 기반 추출 함수

아래 함수들이 위의 기본 함수를 덮어씁니다. `.datajur` 커널에서 `bs4`를 사용할 수 있으므로 이 방식이 더 안정적입니다.

In [ ]:
TEXT_SELECTORS = [
    ".se-main-container",        # 네이버 블로그 스마트에디터 본문
    "#postViewArea",             # 네이버 블로그 구형 본문
    "div[id^='post-view']",      # 네이버 블로그 구형 본문
    ".post_view",                # 블로그/카페 일부 본문
    ".se_component_wrap",        # 구형 스마트에디터
    ".ContentRenderer",          # 네이버 카페 일부 본문
    ".article_viewer",           # 네이버 카페 일부 본문
    ".ArticleContentBox",        # 네이버 카페 일부 본문
    ".article_container",         # 네이버 카페 일부 본문
    ".article_board",             # 네이버 카페 일부 본문
    ".article_view",              # 네이버 카페 일부 본문
    ".article_content",           # 네이버 카페 일부 본문
    ".se-viewer",                 # 네이버 카페 스마트에디터 본문
    "#tbody",                    # 네이버 카페 구형 본문
    "article",
    "main",
]


def remove_noise(soup):
    for tag in soup(["script", "style", "noscript", "svg", "canvas", "iframe"]):
        tag.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    return soup


def soup_text(tag):
    if tag is None:
        return ""
    local = BeautifulSoup(str(tag), "html.parser")
    remove_noise(local)
    return normalize_text(local.get_text("\n"))


def find_iframe_src(html, base_url, iframe_keywords):
    soup = BeautifulSoup(html, "html.parser")
    for iframe in soup.find_all("iframe"):
        haystack = f"{iframe.get('id', '')} {iframe.get('name', '')}".lower()
        if any(keyword.lower() in haystack for keyword in iframe_keywords):
            src = iframe.get("src")
            if src:
                src = src.strip()
                if src.lower().startswith(("javascript:", "about:", "#")):
                    continue
                return urljoin(base_url, src)
    return None


def extract_target_text(html):
    soup = BeautifulSoup(html, "html.parser")
    remove_noise(soup)

    candidates = []
    for selector in TEXT_SELECTORS:
        for tag in soup.select(selector):
            text = soup_text(tag)
            if len(text) >= 20:
                candidates.append(text)

    if candidates:
        return max(candidates, key=len)

    body = soup.body or soup
    return normalize_text(body.get_text("\n"))

## 3. URL 접속 및 원문 수집 함수

In [ ]:
def classify_url(url):
    host = urlparse(url).netloc.lower()
    if "blog.naver.com" in host:
        return "naver_blog"
    if "cafe.naver.com" in host:
        return "naver_cafe"
    return "other"


def fetch_html(session, url):
    if not str(url).lower().startswith(("http://", "https://")):
        raise requests.RequestException(f"unsupported url for requests: {url}")
    response = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    return response.status_code, response.url, response.text


CAFE_RESTRICTED_PATTERNS = [
    "카페 회원만",
    "카페멤버만",
    "멤버만",
    "가입 후",
    "로그인 후",
    "로그인이 필요",
    "권한이 없습니다",
    "접근 권한",
    "등급 이상",
    "게시글이 없습니다",
    "삭제되었거나",
]


def normalize_cafe_url(url):
    parsed = urlparse(url)
    if "cafe.naver.com" not in parsed.netloc.lower():
        return url

    qs = parse_qs(parsed.query)
    iframe_url = (qs.get("iframe_url_utf8") or qs.get("iframe_url") or [None])[0]
    if iframe_url:
        return urljoin(url, iframe_url)

    parts = [part for part in parsed.path.split("/") if part]
    if len(parts) >= 2 and parts[-1].isdigit():
        cafe_name = parts[-2]
        article_id = parts[-1]
        return f"https://cafe.naver.com/{cafe_name}/{article_id}"

    return url


def looks_restricted_cafe(html):
    text = normalize_text(BeautifulSoup(html or "", "html.parser").get_text("\n"))
    return any(pattern in text for pattern in CAFE_RESTRICTED_PATTERNS)


def cafe_restricted_status(html):
    text = normalize_text(BeautifulSoup(html or "", "html.parser").get_text("\n"))
    if "로그인" in text:
        return "restricted_login_required"
    if "가입" in text or "카페 회원" in text or "카페멤버" in text or "멤버만" in text:
        return "restricted_cafe_member_required"
    if "권한" in text or "등급" in text:
        return "restricted_permission_required"
    if "삭제" in text or "게시글이 없습니다" in text:
        return "restricted_deleted_or_missing"
    return "restricted_or_unavailable"


def fetch_full_text(url, session):
    result = {
        "full_text": "",
        "full_text_status": "",
        "full_text_length": 0,
        "full_text_url": "",
        "full_text_collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "full_text_error": "",
    }

    if not url:
        result["full_text_status"] = "missing_url"
        return result

    try:
        page_type = classify_url(url)
        request_url = normalize_cafe_url(url) if page_type == "naver_cafe" else url
        status_code, final_url, html = fetch_html(session, request_url)
        result["full_text_url"] = final_url

        if status_code != 200:
            result["full_text_status"] = f"http_{status_code}"
            return result

        # 네이버 블로그/카페는 본문이 iframe 안에 들어있는 경우가 많습니다.
        iframe_url = None
        if page_type == "naver_blog":
            iframe_url = find_iframe_src(html, final_url, ["mainFrame"])
        elif page_type == "naver_cafe":
            iframe_url = find_iframe_src(html, final_url, ["cafe_main", "mainFrame"])
            if not iframe_url:
                iframe_url = find_iframe_src(html, final_url, ["cafe_main", "mainframe", "cafe"])

        if iframe_url:
            time.sleep(REQUEST_DELAY_SEC)
            iframe_status, iframe_final_url, iframe_html = fetch_html(session, iframe_url)
            result["full_text_url"] = iframe_final_url
            if iframe_status == 200:
                html = iframe_html
            else:
                result["full_text_status"] = f"iframe_http_{iframe_status}"
                return result

        if page_type == "naver_cafe" and looks_restricted_cafe(html):
            text = extract_target_text(html)
            result["full_text"] = text
            result["full_text_length"] = len(text)
            result["full_text_status"] = cafe_restricted_status(html)
            return result

        text = extract_target_text(html)
        result["full_text"] = text
        result["full_text_length"] = len(text)

        if page_type == "naver_cafe" and not text:
            result["full_text_status"] = "empty_text_or_restricted_cafe"
            return result

        if len(text) >= MIN_TEXT_LENGTH:
            result["full_text_status"] = "success"
        elif text:
            result["full_text_status"] = "short_text"
        else:
            result["full_text_status"] = "empty_text"

        return result

    except requests.Timeout:
        result["full_text_status"] = "timeout"
        result["full_text_error"] = "request timeout"
        return result
    except requests.RequestException as e:
        result["full_text_status"] = "request_error"
        result["full_text_error"] = str(e)
        return result
    except Exception as e:
        result["full_text_status"] = "parse_error"
        result["full_text_error"] = str(e)
        return result

## 4. 테스트 수집

전체 CSV를 갱신하기 전에 일부 URL만 테스트합니다. 여기서는 파일을 저장하지 않습니다.

In [ ]:
session = requests.Session()
test_rows = rows[:TEST_ROWS]

for i, row in enumerate(test_rows, start=1):
    url = get_post_link(row)
    print(f"[{i}/{len(test_rows)}] {get_post_source(row)} {url}")
    fulltext_result = fetch_full_text(url, session)
    print("  status:", fulltext_result["full_text_status"])
    print("  length:", fulltext_result["full_text_length"])
    print("  fetched_url:", fulltext_result["full_text_url"])
    preview = fulltext_result["full_text"][:300].replace("\n", " ")
    print("  preview:", preview)
    print()
    time.sleep(REQUEST_DELAY_SEC)

## 4-B. 카페 공개/제한 URL 테스트

제공한 공개 카페 URL과 로그인/가입 제한 카페 URL을 대상으로 본문 추출 가능 여부와 제한 상태 판별이 잘 되는지 확인합니다.

In [ ]:
TEST_PUBLIC_CAFE_URL = "https://cafe.naver.com/fyekfl/51265"
TEST_RESTRICTED_CAFE_URL = "https://cafe.naver.com/statcraftusemap?iframe_url_utf8=%2FArticleRead.nhn%3Fclubid%3D24080865%26articleid%3D10685"

session = requests.Session()
for label, url in [("public_cafe", TEST_PUBLIC_CAFE_URL), ("restricted_cafe", TEST_RESTRICTED_CAFE_URL)]:
    print(f"[{label}] {url}")
    result = fetch_full_text(url, session)
    print("  status:", result["full_text_status"])
    print("  length:", result["full_text_length"])
    print("  fetched_url:", result["full_text_url"])
    print("  error:", result.get("full_text_error", ""))
    print("  preview:", result["full_text"][:500].replace("\n", " "))
    print()
    time.sleep(REQUEST_DELAY_SEC)

## 5. 전체 URL 원문 수집 후 CSV에 병합

아래 셀은 전체 행을 순회합니다. 실행하면 기존 CSV를 백업한 뒤, 원 CSV 파일에 `full_text` 관련 컬럼을 마지막에 추가해서 다시 저장합니다.

In [ ]:
def write_rows(path, rows, fieldnames):
    output_df = pd.DataFrame(rows).reindex(columns=fieldnames).fillna("")
    output_df.to_csv(path, index=False, encoding="utf-8-sig")


def merge_fulltext_columns(rows, fieldnames):
    merged_fieldnames = [name for name in fieldnames if name not in FULLTEXT_COLUMNS]
    merged_fieldnames.extend(FULLTEXT_COLUMNS)
    for row in rows:
        for col in FULLTEXT_COLUMNS:
            row.setdefault(col, "")
    return rows, merged_fieldnames


rows, fieldnames = read_csv_rows(INPUT_CSV)
rows, merged_fieldnames = merge_fulltext_columns(rows, fieldnames)

if not BACKUP_CSV.exists():
    shutil.copy2(INPUT_CSV, BACKUP_CSV)
    print(f"백업 생성: {BACKUP_CSV}")
else:
    print(f"기존 백업 유지: {BACKUP_CSV}")

session = requests.Session()
success_count = 0
skip_count = 0
fail_count = 0

for i, row in enumerate(rows, start=1):
    if SKIP_EXISTING_FULL_TEXT and get_existing_full_text(row):
        skip_count += 1
        continue

    url = get_post_link(row)
    print(f"[{i}/{len(rows)}] {get_post_source(row)} {url}")
    fulltext_result = fetch_full_text(url, session)
    row.update(fulltext_result)

    if fulltext_result["full_text_status"] == "success":
        success_count += 1
    else:
        fail_count += 1

    print(
        f"  status={fulltext_result['full_text_status']} "
        f"length={fulltext_result['full_text_length']}"
    )

    if i % SAVE_EVERY == 0:
        write_rows(TEMP_OUTPUT_CSV, rows, merged_fieldnames)
        TEMP_OUTPUT_CSV.replace(INPUT_CSV)
        print(f"  중간 저장 완료: {i}행 처리")

    time.sleep(REQUEST_DELAY_SEC)

write_rows(TEMP_OUTPUT_CSV, rows, merged_fieldnames)
TEMP_OUTPUT_CSV.replace(INPUT_CSV)

print("\n원문 보강 완료")
print(f"성공: {success_count:,}건")
print(f"실패/짧은 텍스트: {fail_count:,}건")
print(f"기존 full_text로 스킵: {skip_count:,}건")
print(f"갱신 파일: {INPUT_CSV}")
print(f"백업 파일: {BACKUP_CSV}")

## 6. 결과 확인

In [ ]:
updated_rows, updated_fieldnames = read_csv_rows(INPUT_CSV)
print(f"갱신 후 행 수: {len(updated_rows):,}")
print(f"마지막 컬럼들: {updated_fieldnames[-len(FULLTEXT_COLUMNS):]}")

status_counts = {}
for row in updated_rows:
    status = row.get("full_text_status", "")
    status_counts[status] = status_counts.get(status, 0) + 1

print("\nfull_text_status 분포")
for status, count in sorted(status_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{status or '(blank)'}: {count:,}")

for row in updated_rows[:3]:
    print("\n---")
    print(get_post_title(row))
    print(row.get("full_text_status", ""), row.get("full_text_length", ""))
    print(row.get("full_text", "")[:500].replace("\n", " "))

## 7. 블로그 댓글 수집 함수

네이버 블로그 댓글은 `apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json`, 공개 카페 댓글은 `article.cafe.naver.com/gw/v4/cafes/{cafe_id}/articles/{article_id}/comments/pages/{page}` 요청으로 불러오는 경우가 많습니다.

공개 접근 가능한 카페글은 댓글 수집을 시도하고, 로그인/가입/권한 제한 글은 상태값으로 기록합니다.

In [ ]:
COMMENT_PAGE_SIZE = 50
COMMENT_REPLY_PAGE_SIZE = 100
MAX_COMMENT_PAGES_PER_POST = 20
COMMENT_REQUEST_DELAY_SEC = 0.5
COMMENT_SAVE_EVERY_POSTS = 20

COMMENT_COLUMNS = [
    "comments_file_key",
    "post_row_index",
    "post_link",
    "source",
    "post_title",
    "object_id",
    "group_id",
    "comment_no",
    "parent_comment_no",
    "is_reply",
    "reply_level",
    "comment_text",
    "user_name",
    "masked_user_name",
    "profile_user_id",
    "reg_time",
    "reg_time_gmt",
    "mod_time",
    "sympathy_count",
    "antipathy_count",
    "reply_count",
    "best",
    "visible",
    "deleted",
    "blind",
    "secret",
    "comment_fetch_status",
]

In [ ]:
def make_comments_file_key(link):
    return hashlib.md5(str(link).encode("utf-8")).hexdigest()[:16]


def parse_jsonp_response(text):
    raw = (text or "").strip()
    if raw.startswith("/**/"):
        raw = raw[4:].strip()
    start = raw.find("(")
    end = raw.rfind(")")
    if start == -1 or end == -1 or end <= start:
        return json.loads(raw)
    return json.loads(raw[start + 1:end])


def parse_blog_id_log_no(url):
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    blog_id = (qs.get("blogId") or qs.get("blogId".lower()) or [None])[0]
    log_no = (qs.get("logNo") or qs.get("logNo".lower()) or [None])[0]

    parts = [part for part in parsed.path.split("/") if part]
    if not blog_id and len(parts) >= 1:
        blog_id = parts[0]
    if not log_no and len(parts) >= 2 and parts[1].isdigit():
        log_no = parts[1]

    return blog_id, log_no


def find_first_regex(patterns, text):
    for pattern in patterns:
        match = re.search(pattern, text or "")
        if match:
            return match.group(1)
    return None


def get_blog_comment_context(url, session):
    status_code, final_url, html = fetch_html(session, url)
    if status_code != 200:
        return {"status": f"post_http_{status_code}", "referer": final_url}

    iframe_url = find_iframe_src(html, final_url, ["mainFrame"])
    iframe_html = html
    referer = final_url
    if iframe_url:
        iframe_status, iframe_final_url, iframe_html = fetch_html(session, iframe_url)
        referer = iframe_final_url
        if iframe_status != 200:
            return {"status": f"iframe_http_{iframe_status}", "referer": iframe_final_url}

    blog_id, log_no = parse_blog_id_log_no(referer)
    if not log_no:
        _, log_no = parse_blog_id_log_no(url)

    object_id = find_first_regex([
        r"objectId[\"']?\s*[:=]\s*[\"']([^\"']+)[\"']",
        r"commentObjectId[\"']?\s*[:=]\s*[\"']([^\"']+)[\"']",
        r"objectId=([0-9_]+)",
    ], iframe_html)

    blog_no = find_first_regex([
        r"blogNo[\"']?\s*[:=]\s*[\"']?(\d+)",
        r"blogNo=(\d+)",
        r"groupId[\"']?\s*[:=]\s*[\"']?(\d+)",
    ], iframe_html)

    if object_id and not blog_no:
        blog_no = object_id.split("_")[0]

    # 네이버 블로그 댓글 objectId는 대체로 blogNo_201_logNo 형식입니다.
    if not object_id and blog_no and log_no:
        object_id = f"{blog_no}_201_{log_no}"

    if not object_id or not blog_no:
        return {
            "status": "missing_blog_comment_context",
            "referer": referer,
            "blog_id": blog_id or "",
            "log_no": log_no or "",
            "object_id": object_id or "",
            "group_id": blog_no or "",
        }

    return {
        "status": "success",
        "referer": referer,
        "blog_id": blog_id or "",
        "log_no": log_no or "",
        "object_id": object_id,
        "group_id": blog_no,
    }

In [ ]:
def flatten_comment_items(comment_items):
    flat = []
    for item in comment_items or []:
        flat.append(item)
        reply_list = item.get("replyList") or []
        if reply_list:
            flat.extend(flatten_comment_items(reply_list))
    return flat


def normalize_comment(item, row, row_index, comments_file_key, context, fetch_status):
    comment_no = str(item.get("commentNo", ""))
    parent_comment_no = str(item.get("parentCommentNo", ""))
    reply_level = item.get("replyLevel", "")
    is_reply = bool(parent_comment_no and parent_comment_no != comment_no) or str(reply_level) not in {"", "1"}

    return {
        "comments_file_key": comments_file_key,
        "post_row_index": row_index,
        "post_link": get_post_link(row),
        "source": get_post_source(row),
        "post_title": get_post_title(row),
        "object_id": context.get("object_id", ""),
        "group_id": context.get("group_id", ""),
        "comment_no": comment_no,
        "parent_comment_no": parent_comment_no,
        "is_reply": is_reply,
        "reply_level": reply_level,
        "comment_text": item.get("contents", ""),
        "user_name": item.get("userName", ""),
        "masked_user_name": item.get("maskedUserName", ""),
        "profile_user_id": item.get("profileUserId", ""),
        "reg_time": item.get("regTime", ""),
        "reg_time_gmt": item.get("regTimeGmt", ""),
        "mod_time": item.get("modTime", ""),
        "sympathy_count": item.get("sympathyCount", 0),
        "antipathy_count": item.get("antipathyCount", 0),
        "reply_count": item.get("replyCount", 0),
        "best": item.get("best", False),
        "visible": item.get("visible", ""),
        "deleted": item.get("deleted", ""),
        "blind": item.get("blind", ""),
        "secret": item.get("secret", ""),
        "comment_fetch_status": fetch_status,
    }


def fetch_blog_comment_page(session, context, page):
    callback = f"jQuery{int(time.time() * 1000)}"
    params = {
        "ticket": "blog",
        "templateId": "default",
        "pool": "blogid",
        "_callback": callback,
        "lang": "ko",
        "country": "",
        "objectId": context["object_id"],
        "categoryId": "",
        "pageSize": COMMENT_PAGE_SIZE,
        "indexSize": 10,
        "groupId": context["group_id"],
        "listType": "OBJECT",
        "pageType": "default",
        "page": page,
        "initialize": "true" if page == 1 else "false",
        "followSize": 5,
        "userType": "",
        "useAltSort": "true",
        "replyPageSize": COMMENT_REPLY_PAGE_SIZE,
        "showReply": "true",
        "_": int(time.time() * 1000),
    }
    headers = dict(HEADERS)
    headers.update({"Referer": context.get("referer", "https://blog.naver.com/"), "Accept": "*/*"})
    url = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"
    response = session.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return parse_jsonp_response(response.text)


def fetch_blog_comments(row, row_index, session):
    link = get_post_link(row)
    comments_file_key = make_comments_file_key(link)
    context = get_blog_comment_context(link, session)
    if context.get("status") != "success":
        return [], context.get("status", "context_error"), comments_file_key

    all_comments = []
    seen_comment_nos = set()
    fetch_status = "success"

    for page in range(1, MAX_COMMENT_PAGES_PER_POST + 1):
        data = fetch_blog_comment_page(session, context, page)
        if not data.get("success"):
            fetch_status = f"api_{data.get('code', 'error')}"
            break

        comment_items = data.get("result", {}).get("commentList", []) or []
        flat_items = flatten_comment_items(comment_items)
        if not flat_items:
            break

        for item in flat_items:
            comment_no = str(item.get("commentNo", ""))
            if comment_no and comment_no in seen_comment_nos:
                continue
            seen_comment_nos.add(comment_no)
            all_comments.append(
                normalize_comment(item, row, row_index, comments_file_key, context, fetch_status)
            )

        if len(comment_items) < COMMENT_PAGE_SIZE:
            break
        time.sleep(COMMENT_REQUEST_DELAY_SEC)

    return all_comments, fetch_status, comments_file_key

### 공개 카페 댓글 수집 함수

공개 카페글에서 `cafe_id`와 `article_id`를 찾은 뒤 카페 댓글 API를 호출합니다. 로그인/가입/권한 제한 글은 수집하지 않고 상태값만 남깁니다.

In [ ]:
def timestamp_ms_to_iso(value):
    if value in (None, ""):
        return ""
    try:
        return datetime.fromtimestamp(int(value) / 1000).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


def parse_cafe_ids_from_url(url):
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    cafe_id = (qs.get("clubid") or qs.get("cafeId") or qs.get("cafeid") or [None])[0]
    article_id = (qs.get("articleid") or qs.get("articleId") or [None])[0]

    iframe_url = (qs.get("iframe_url_utf8") or qs.get("iframe_url") or [None])[0]
    if iframe_url:
        iframe_cafe_id, iframe_article_id = parse_cafe_ids_from_url(urljoin(url, iframe_url))
        cafe_id = cafe_id or iframe_cafe_id
        article_id = article_id or iframe_article_id

    match = re.search(r"/ca-fe/cafes/(\d+)/articles/(\d+)", parsed.path)
    if match:
        cafe_id = cafe_id or match.group(1)
        article_id = article_id or match.group(2)

    parts = [part for part in parsed.path.split("/") if part]
    if not article_id and len(parts) >= 2 and parts[-1].isdigit():
        article_id = parts[-1]

    return cafe_id, article_id


def get_cafe_comment_context(url, session):
    request_url = normalize_cafe_url(url)
    status_code, final_url, html = fetch_html(session, request_url)
    if status_code != 200:
        return {"status": f"post_http_{status_code}", "referer": final_url}

    if looks_restricted_cafe(html):
        return {"status": cafe_restricted_status(html), "referer": final_url}

    cafe_id, article_id = parse_cafe_ids_from_url(final_url)
    if not cafe_id or not article_id:
        html_cafe_id = find_first_regex([
            r'"cafe"\s*:\s*\{\s*"id"\s*:\s*(\d+)',
            r'clubid[=:\"\s]+(\d+)',
            r'clubId[=:\"\s]+(\d+)',
            r'cafeId[=:\"\s]+(\d+)',
        ], html)
        html_article_id = find_first_regex([
            r'"article"\s*:\s*\{\s*"id"\s*:\s*(\d+)',
            r'articleid[=:\"\s]+(\d+)',
            r'articleId[=:\"\s]+(\d+)',
        ], html)
        cafe_id = cafe_id or html_cafe_id
        article_id = article_id or html_article_id

    if not cafe_id or not article_id:
        return {
            "status": "missing_cafe_comment_context",
            "referer": final_url,
            "cafe_id": cafe_id or "",
            "article_id": article_id or "",
        }

    return {
        "status": "success",
        "referer": final_url,
        "cafe_id": str(cafe_id),
        "article_id": str(article_id),
        "object_id": f"{cafe_id}_{article_id}",
        "group_id": str(cafe_id),
    }


In [ ]:
def fetch_cafe_comment_page(session, context, page):
    url = (
        f"https://article.cafe.naver.com/gw/v4/cafes/{context['cafe_id']}"
        f"/articles/{context['article_id']}/comments/pages/{page}"
    )
    params = {"requestFrom": "A", "orderBy": "asc"}
    headers = dict(HEADERS)
    headers.update({
        "Accept": "application/json, text/plain, */*",
        "Origin": "https://cafe.naver.com",
        "Referer": context.get("referer", "https://cafe.naver.com/"),
        "X-Cafe-Product": "pc",
    })
    response = session.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def get_cafe_comment_items(data):
    if not isinstance(data, dict):
        return []
    comments = data.get("comments") or data.get("result", {}).get("comments") or {}
    return comments.get("items") or []


def get_cafe_comment_total(data):
    article = data.get("article") or data.get("result", {}).get("article") or {}
    total = article.get("commentCount")
    try:
        return int(total)
    except Exception:
        return None


def normalize_cafe_comment(item, row, row_index, comments_file_key, context, fetch_status):
    comment_no = str(item.get("id", ""))
    parent_comment_no = str(item.get("refId", ""))
    is_reply = bool(item.get("isRef", False)) or bool(parent_comment_no and parent_comment_no != comment_no)
    writer = item.get("writer") or {}

    return {
        "comments_file_key": comments_file_key,
        "post_row_index": row_index,
        "post_link": get_post_link(row),
        "source": get_post_source(row),
        "post_title": get_post_title(row),
        "object_id": context.get("object_id", ""),
        "group_id": context.get("group_id", ""),
        "comment_no": comment_no,
        "parent_comment_no": parent_comment_no,
        "is_reply": is_reply,
        "reply_level": 2 if is_reply else 1,
        "comment_text": item.get("content", ""),
        "user_name": writer.get("nick", ""),
        "masked_user_name": writer.get("nick", ""),
        "profile_user_id": writer.get("memberKey", ""),
        "reg_time": timestamp_ms_to_iso(item.get("updateDate", "")),
        "reg_time_gmt": "",
        "mod_time": timestamp_ms_to_iso(item.get("updateDate", "")),
        "sympathy_count": item.get("sympathyCount", item.get("likeCount", 0)),
        "antipathy_count": item.get("antipathyCount", 0),
        "reply_count": item.get("replyCount", 0),
        "best": item.get("bestComment", False),
        "visible": not item.get("isDeleted", False),
        "deleted": item.get("isDeleted", False),
        "blind": item.get("cleanBotDetected", False),
        "secret": False,
        "comment_fetch_status": fetch_status,
    }


def fetch_cafe_comments(row, row_index, session):
    link = get_post_link(row)
    comments_file_key = make_comments_file_key(link)
    context = get_cafe_comment_context(link, session)
    if context.get("status") != "success":
        return [], context.get("status", "context_error"), comments_file_key

    all_comments = []
    seen_comment_nos = set()
    fetch_status = "success"
    total = None

    for page in range(1, MAX_COMMENT_PAGES_PER_POST + 1):
        data = fetch_cafe_comment_page(session, context, page)
        items = get_cafe_comment_items(data)
        if total is None:
            total = get_cafe_comment_total(data)

        if not items:
            break

        for item in items:
            comment_no = str(item.get("id", ""))
            if comment_no and comment_no in seen_comment_nos:
                continue
            seen_comment_nos.add(comment_no)
            all_comments.append(
                normalize_cafe_comment(item, row, row_index, comments_file_key, context, fetch_status)
            )

        if total is not None and len(seen_comment_nos) >= total:
            break
        time.sleep(COMMENT_REQUEST_DELAY_SEC)

    return all_comments, fetch_status, comments_file_key


## 8. 블로그 댓글 테스트 수집

제공한 예시 URL 또는 CSV 안의 블로그 URL 1개로 댓글 수집이 되는지 확인합니다.

In [ ]:
TEST_BLOG_COMMENT_URL = "https://blog.naver.com/kukuhihi_/224262038139"

test_comment_row = {
    "source": "blog",
    "link": TEST_BLOG_COMMENT_URL,
    "title": "댓글 테스트용 블로그 글",
}

session = requests.Session()
test_comments, test_status, test_key = fetch_blog_comments(test_comment_row, 0, session)
print(f"status={test_status}, comments={len(test_comments)}, key={test_key}")
pd.DataFrame(test_comments).head(10)

## 8-B. 공개 카페 댓글 테스트 수집

제공한 공개 카페 URL로 댓글 API 수집이 되는지 확인합니다.

In [ ]:
TEST_CAFE_COMMENT_URL = "https://cafe.naver.com/fyekfl/51265"

test_cafe_comment_row = {
    "source": "cafearticle",
    "link": TEST_CAFE_COMMENT_URL,
    "title": "댓글 테스트용 공개 카페글",
}

session = requests.Session()
test_cafe_comments, test_cafe_status, test_cafe_key = fetch_cafe_comments(test_cafe_comment_row, 0, session)
print(f"status={test_cafe_status}, comments={len(test_cafe_comments)}, key={test_cafe_key}")
pd.DataFrame(test_cafe_comments).head(20)


## 9. 전체 블로그/공개 카페 댓글 수집 및 병합

댓글은 현재 주제별 파일인 `crawl_comments_{CORE_KEYWORD}.csv` 형태로 별도 저장하고, 기존 게시글 CSV에는 댓글 수집 요약 컬럼만 마지막에 추가합니다. 블로그와 공개 접근 가능한 카페글은 댓글 수집을 시도하고, 제한 카페글은 상태값으로 기록합니다.

중간에 실행을 멈췄다가 다시 실행하면 기존 댓글 CSV를 먼저 읽고, 이미 `comment_fetch_status`가 기록된 게시글은 건너뜁니다. 새로 수집한 댓글은 기존 댓글 CSV 뒤에 이어서 저장됩니다.

In [ ]:
def write_post_rows_with_comment_summary(path, rows, fieldnames):
    merged_fieldnames = [name for name in fieldnames if name not in COMMENT_SUMMARY_COLUMNS]
    merged_fieldnames.extend(COMMENT_SUMMARY_COLUMNS)
    output_df = pd.DataFrame(rows).reindex(columns=merged_fieldnames).fillna("")
    output_df.to_csv(path, index=False, encoding="utf-8-sig")
    return merged_fieldnames


def read_existing_comments(path):
    if not path.exists():
        return []
    comments_df = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")
    comments_df = comments_df.reindex(columns=COMMENT_COLUMNS).fillna("")
    return comments_df.to_dict("records")


def comment_identity(comment_row):
    post_link = str(comment_row.get("post_link", "")).strip()
    comment_no = str(comment_row.get("comment_no", "")).strip()
    if post_link or comment_no:
        return post_link, comment_no
    return (
        str(comment_row.get("comments_file_key", "")).strip(),
        str(comment_row.get("comment_text", "")).strip(),
        str(comment_row.get("reg_time", "")).strip(),
        str(comment_row.get("user_name", "")).strip(),
    )


def append_unique_comments(all_comment_rows, new_comment_rows, seen_comment_keys):
    added = 0
    for comment_row in new_comment_rows:
        key = comment_identity(comment_row)
        if key in seen_comment_keys:
            continue
        seen_comment_keys.add(key)
        all_comment_rows.append(comment_row)
        added += 1
    return added


def write_comments(path, comment_rows):
    comments_df = pd.DataFrame(comment_rows)
    if comments_df.empty:
        comments_df = pd.DataFrame(columns=COMMENT_COLUMNS)
    else:
        comments_df = comments_df.reindex(columns=COMMENT_COLUMNS).fillna("")
    comments_df.to_csv(path, index=False, encoding="utf-8-sig")


rows, fieldnames = read_csv_rows(INPUT_CSV)
for row in rows:
    for col in COMMENT_SUMMARY_COLUMNS:
        row.setdefault(col, "")

session = requests.Session()
all_comment_rows = read_existing_comments(COMMENTS_CSV)
seen_comment_keys = {comment_identity(comment_row) for comment_row in all_comment_rows}
print(f"기존 댓글 파일: {COMMENTS_CSV}")
print(f"기존 댓글 행 수: {len(all_comment_rows):,}건")

blog_attempts = 0
blog_success = 0
blog_fail = 0
cafe_attempts = 0
cafe_success = 0
cafe_fail = 0
other_skip = 0
resume_skip = 0
new_comment_count = 0

for row_index, row in enumerate(rows):
    source = get_post_source(row)
    link = get_post_link(row)
    row.setdefault("comments_file_key", make_comments_file_key(link))

    if SKIP_EXISTING_COMMENTS and str(row.get("comment_fetch_status", "")).strip():
        resume_skip += 1
        continue

    row["comments_file_key"] = make_comments_file_key(link)
    row["comment_collected_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if source == "blog":
        blog_attempts += 1
        source_label = "blog"
        fetcher = fetch_blog_comments
    elif source == "cafearticle":
        cafe_attempts += 1
        source_label = "cafearticle"
        fetcher = fetch_cafe_comments
    else:
        row["comment_count_collected"] = 0
        row["comment_fetch_status"] = "not_supported_source"
        other_skip += 1
        continue

    print(f"[{row_index + 1}/{len(rows)}] {source_label} comments {link}")

    try:
        comment_rows, status, comments_file_key = fetcher(row, row_index, session)
    except requests.RequestException as e:
        comment_rows, status, comments_file_key = [], f"request_error: {e}", row["comments_file_key"]
    except Exception as e:
        comment_rows, status, comments_file_key = [], f"comment_error: {e}", row["comments_file_key"]

    added_count = append_unique_comments(all_comment_rows, comment_rows, seen_comment_keys)
    new_comment_count += added_count

    row["comments_file_key"] = comments_file_key
    row["comment_count_collected"] = len(comment_rows)
    row["comment_fetch_status"] = status

    if source == "blog":
        if status == "success":
            blog_success += 1
        else:
            blog_fail += 1
    elif source == "cafearticle":
        if status == "success":
            cafe_success += 1
        else:
            cafe_fail += 1

    print(f"  status={status}, comments={len(comment_rows)}, added={added_count}")

    if (blog_attempts + cafe_attempts) % COMMENT_SAVE_EVERY_POSTS == 0:
        write_comments(COMMENTS_TEMP_CSV, all_comment_rows)
        COMMENTS_TEMP_CSV.replace(COMMENTS_CSV)
        write_post_rows_with_comment_summary(INPUT_CSV, rows, fieldnames)
        print(
            f"  중간 저장: attempts={blog_attempts + cafe_attempts}, "
            f"existing+new_comments={len(all_comment_rows)}, new_added={new_comment_count}, skipped={resume_skip}"
        )

    time.sleep(COMMENT_REQUEST_DELAY_SEC)

write_comments(COMMENTS_TEMP_CSV, all_comment_rows)
COMMENTS_TEMP_CSV.replace(COMMENTS_CSV)
updated_fieldnames = write_post_rows_with_comment_summary(INPUT_CSV, rows, fieldnames)

print("\n댓글 수집 완료")
print(f"기존 처리 행 스킵: {resume_skip:,}건")
print(f"블로그 신규 시도: {blog_attempts:,}건")
print(f"블로그 success status: {blog_success:,}건")
print(f"블로그 실패/기타 status: {blog_fail:,}건")
print(f"카페 신규 시도: {cafe_attempts:,}건")
print(f"카페 success status: {cafe_success:,}건")
print(f"카페 실패/기타 status: {cafe_fail:,}건")
print(f"기타 source 스킵: {other_skip:,}건")
print(f"이번 실행에서 새로 추가된 댓글 행 수: {new_comment_count:,}건")
print(f"댓글 파일 전체 행 수: {len(all_comment_rows):,}건")
print(f"댓글 파일: {COMMENTS_CSV}")
print(f"게시글 CSV 갱신: {INPUT_CSV}")

## 10. 댓글 수집 결과 확인

In [ ]:
comments_df = pd.read_csv(COMMENTS_CSV, encoding="utf-8-sig", dtype=str).fillna("")
posts_df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", dtype=str).fillna("")

print(f"댓글 데이터: {len(comments_df):,}행")
print(f"댓글 컬럼: {list(comments_df.columns)}")
print("\ncomment_fetch_status 분포")
print(posts_df.get("comment_fetch_status", pd.Series(dtype=str)).value_counts(dropna=False))

comments_df.head(20)

## 11. 입력 URL 중복 확인

URL 기준 중복 제거는 1단계 `test_crawl_teamproject.ipynb`에서 주제별 CSV 저장 전에 수행합니다. 여기서는 현재 `CORE_KEYWORD` 입력 파일이 실제로 중복 제거된 상태인지 확인만 합니다.

In [ ]:
posts_df_for_check = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", dtype=str).fillna("")
url_column = next((col for col in ["link", "게시글URL"] if col in posts_df_for_check.columns), None)

if url_column is None:
    raise ValueError(f"URL 컬럼이 없습니다. 현재 컬럼: {list(posts_df_for_check.columns)}")

duplicate_mask = posts_df_for_check.duplicated(subset=[url_column], keep=False)
duplicate_rows = int(duplicate_mask.sum())
duplicate_groups = int(posts_df_for_check.loc[duplicate_mask, url_column].nunique())

print(f"처리 주제: {CORE_KEYWORD}")
print(f"입력 파일: {INPUT_CSV}")
print(f"URL 기준 컬럼: {url_column}")
print(f"전체 행 수: {len(posts_df_for_check):,}")
print(f"고유 URL 수: {posts_df_for_check[url_column].nunique():,}")
print(f"중복 URL 포함 행 수: {duplicate_rows:,}")
print(f"중복 URL 그룹 수: {duplicate_groups:,}")

if duplicate_rows:
    print("중복이 남아 있습니다. 1단계 노트북에서 현재 주제 CSV를 다시 생성하세요.")
    display_columns = [col for col in [url_column, "title", "게시글제목", "search_keywords", "검색키워드조합", "search_query", "검색쿼리"] if col in posts_df_for_check.columns]
    display(posts_df_for_check.loc[duplicate_mask, display_columns].head(20))
else:
    print("URL 기준 중복이 없습니다. 이 파일을 대상으로 원문/댓글 수집을 진행하면 됩니다.")

## 12. 댓글 URL+댓글ID 기준 중복 제거 데이터 만들기

댓글 데이터에서 같은 게시글의 같은 댓글은 1행만 남깁니다. 원본 댓글 CSV는 유지하고, 중복 제거 결과를 새 파일로 저장합니다.

In [ ]:
COMMENTS_DEDUP_INPUT_CSV = COMMENTS_CSV
COMMENTS_DEDUP_OUTPUT_CSV = BASE_DIR / f"{COMMENTS_FILE_STEM}_dedup_by_url_comment_id.csv"

COMMENT_DEDUP_KEY_CANDIDATES = [
    ["게시글URL", "댓글ID"],
    ["post_link", "comment_no"],
]

comments_original = pd.read_csv(COMMENTS_DEDUP_INPUT_CSV, encoding="utf-8-sig", dtype=str).fillna("")

comment_dedup_key_columns = next(
    (columns for columns in COMMENT_DEDUP_KEY_CANDIDATES if all(col in comments_original.columns for col in columns)),
    None,
)
if comment_dedup_key_columns is None:
    raise ValueError(
        "중복 제거 기준 컬럼이 없습니다. "
        f"후보: {COMMENT_DEDUP_KEY_CANDIDATES}, 현재 컬럼: {list(comments_original.columns)}"
    )

comments_dedup = (
    comments_original
    .drop_duplicates(subset=comment_dedup_key_columns, keep="first")
    .reset_index(drop=True)
)

comments_dedup.to_csv(COMMENTS_DEDUP_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"중복 제거 기준 컬럼: {comment_dedup_key_columns}")
print(f"원본 댓글 행 수: {len(comments_original):,}")
print(f"중복 제거 후 댓글 행 수: {len(comments_dedup):,}")
print(f"제거된 중복 댓글 행 수: {len(comments_original) - len(comments_dedup):,}")
print(f"저장 완료: {COMMENTS_DEDUP_OUTPUT_CSV}")

preview_columns = [
    col for col in [
        "게시글URL", "post_link", "댓글ID", "comment_no", "대댓글여부", "is_reply",
        "부모댓글ID", "parent_comment_no", "댓글본문", "comment_text",
    ]
    if col in comments_dedup.columns
]
comments_dedup[preview_columns].head(10)